In [101]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [102]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [103]:
import os
import re
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [104]:
save_dir = "models"

In [29]:
_ = torch.manual_seed(123)

In [105]:
def separate_dots(s):
    s = re.sub(r"\.",".", s)
    return s.strip()

text_data = [separate_dots(x) for x in data]

In [106]:
example_data = text_data[0]
print(example_data)

She composes songs and practices piano daily.


In [107]:
vocab =set()

for text in text_data:
    for word in text.split():
        vocab.add(word)

vocab = sorted(vocab)
print(vocab)
print(len(vocab))

['He', 'She', 'and', 'art', 'books', 'builds', 'caves.', 'climbs', 'collaborates', 'complex', 'composes', 'creative', 'curates', 'daily.', 'designs', 'digital', 'documents', 'every', 'everyday', 'exhibitions', 'experiments', 'explores', 'fairs.', 'filmmakers.', 'for', 'friends.', 'harmonies', 'her', 'in', 'jewelry', 'local', 'maps', 'marathons.', 'models.', 'mountains', 'music', 'music.', 'navigation', 'nearby', 'newspaper', 'novel', 'novels.', 'organizes', 'participates', 'photography', 'piano', 'practices', 'projects.', 'puzzles.', 'reads', 'regularly.', 'rhythms', 'science', 'small', 'solves', 'songs', 'soundtracks', 'stars.', 'studies', 'teaches', 'the', 'trains', 'trips.', 'tunes', 'using', 'weekend.', 'wildlife', 'with', 'wooden', 'writes']
70


In [108]:
file_name = "vocab.json"
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, file_name), "w") as f:
    json.dump(vocab, f)

In [109]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: He
1: She
2: and
3: art
4: books
5: builds
6: caves.
7: climbs
8: collaborates
9: complex
10: composes
11: creative
12: curates
13: daily.
14: designs
15: digital
16: documents
17: every
18: everyday
19: exhibitions
20: experiments
21: explores
22: fairs.
23: filmmakers.
24: for
25: friends.
26: harmonies
27: her
28: in
29: jewelry
30: local
31: maps
32: marathons.
33: models.
34: mountains
35: music
36: music.
37: navigation
38: nearby
39: newspaper
40: novel
41: novels.
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects.
48: puzzles.
49: reads
50: regularly.
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars.
58: studies
59: teaches
60: the
61: trains
62: trips.
63: tunes
64: using
65: weekend.
66: wildlife
67: with
68: wooden
69: writes


In [110]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

In [111]:
def text_to_token_ids(text, word_to_idx):
    tokens =text.split()
    return [word_to_idx[word] for word in tokens]

def token_ids_to_text(token_ids,idx_to_word):
    return " ".join([idx_to_word[idx] for idx in token_ids])

In [112]:
print(example_data)

She composes songs and practices piano daily.


In [113]:
text_to_token_ids(example_data,word_to_idx)

[1, 10, 55, 2, 46, 45, 13]

In [114]:
text_to_token_ids(example_data,word_to_idx)[:CONTEXT_LEN+1]

[1, 10, 55, 2, 46, 45, 13]

In [115]:
token_ids = text_to_token_ids(example_data,word_to_idx)[:CONTEXT_LEN]

In [180]:
class LLMDataset(Dataset):

    def __init__(self, text_data, word_to_idx, max_len):
        self.text_data = text_data
        self.word_to_idx = word_to_idx
        self.max_len = max_len

    def __len__(self):
        return len(self.text_data)
    
    def __getitem__(self, idx):
        # 🔹 Step 1: get token ids (LIST, not tensor yet)
        tokens = text_to_token_ids(self.text_data[idx], self.word_to_idx)

        # 🔹 Step 2: truncate
        tokens = tokens[:self.max_len + 1]

        # 🔹 Step 3: pad (IMPORTANT FIX)
        if len(tokens) < self.max_len + 1:
            tokens = tokens + [0] * (self.max_len + 1 - len(tokens))

        # 🔹 Step 4: convert to tensor
        tokens = torch.tensor(tokens, dtype=torch.long)

        # 🔹 Step 5: create input/target
        x = tokens[:-1]   # (max_len)
        y = tokens[1:]    # (max_len)

        return x, y


# dataset + dataloader
train_dataset = LLMDataset(text_data, word_to_idx, CONTEXT_LEN)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(train_dataloader)
print(len(train_dataloader))


        

4


In [181]:
example_input_output = next(iter(train_dataloader))
print(example_input_output)

[tensor([[ 0, 49,  4,  2,  5, 53],
        [ 1, 10, 26,  2, 20, 67],
        [ 0, 49, 18,  2, 16, 66],
        [ 1, 10, 55,  2, 12,  3]]), tensor([[49,  4,  2,  5, 53, 68],
        [10, 26,  2, 20, 67, 15],
        [49, 18,  2, 16, 66, 44],
        [10, 55,  2, 12,  3, 47]])]


In [182]:
example_input_output[0][0]


tensor([ 0, 49,  4,  2,  5, 53])

In [183]:
example_input_output[1][0]

tensor([49,  4,  2,  5, 53, 68])

In [184]:
class Attention(nn.Module):
    def __init__(self, d_in, d_out, context_len):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out,bias = False)
        self.W_key = nn.Linear(d_in, d_out,bias = False)
        self.W_value = nn.Linear(d_in, d_out,bias = False)
       
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )
    def forward(self, x,return_weights=False):
        _,num_tokens, _ = x.shape
        queries = self.W_query(x)
        Keys = self.W_key(x)
        Values = self.W_value(x)



        attn_scores = queries @ Keys.transpose(-2, -1) 
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / (self.d_out ** 0.5), dim=-1)
        context_vec = attn_weights @ Values

        if return_weights:
            return context_vec, attn_weights

    
        return context_vec

In [185]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = Attention(
            d_in= EMB_DIM, 
            d_out= EMB_DIM, 
            context_len = CONTEXT_LEN
    )

    def forward(self, x):
        shortcut = x
        x = self.att(x)
        x = x + shortcut
        
        return x

In [186]:
class GPTMOdel(nn.Module):
    def __init__(self):
        super().__init__()

        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)

        self.trm_block1 = TransformerBlock()
        self.trm_block2 = TransformerBlock()
        self.trm_block3 = TransformerBlock()
        self.trm_block4 = TransformerBlock()
        self.trm_block5 = TransformerBlock()
        self.trm_block6 = TransformerBlock()

        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias = False)

       

    def forward(self, in_idx):
        _, seq_len = in_idx.shape

        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len))
        x = tok_emb + pos_emb

        x = self.trm_block1(x)
        x = self.trm_block2(x)
        x = self.trm_block3(x)
        x = self.trm_block4(x)
        x = self.trm_block5(x)
        x = self.trm_block6(x)
        logits = self.output_layer(x)
        return logits
      

In [187]:
model = GPTMOdel()
optimizer = torch.optim.AdamW(model.parameters(), lr= 0.0003,weight_decay= 0.1)
loss_fn = nn.CrossEntropyLoss()

In [188]:
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_parameters}")

Number of trainable parameters: 13872


In [189]:
total_steps = len(train_dataloader) * EPOCHS
total_param = sum(p.numel() for p in model.parameters())
print(f"Total training steps: {total_steps}")
print(f"Total parameters in the model: {total_param}")

Total training steps: 400
Total parameters in the model: 13872


In [190]:
param_count = defaultdict(int)

for name, param in model.named_parameters():
    top_level = name.split(".")[0]
    param_count[top_level] += param.numel()

for k, v in param_count.items():
    print(f"{k:20s}: {v:,} parameters")    

tok_emb             : 1,680 parameters
pos_emb             : 144 parameters
trm_block1          : 1,728 parameters
trm_block2          : 1,728 parameters
trm_block3          : 1,728 parameters
trm_block4          : 1,728 parameters
trm_block5          : 1,728 parameters
trm_block6          : 1,728 parameters
output_layer        : 1,680 parameters


In [191]:
def print_module_params(module, indent=0):
    for name, child in module.named_children():
        print(" " * indent + f"[{name}]")
       
        for p_name, param in child.named_parameters(recurse=False):
            print(" " * (indent + 2) + f"{p_name}: {param.numel():,} parameters")
        print_module_params(child, indent + 2)
    

In [192]:
print_module_params(model.trm_block1)

[att]
  [W_query]
    weight: 576 parameters
  [W_key]
    weight: 576 parameters
  [W_value]
    weight: 576 parameters


In [193]:
print_module_params(model.trm_block6)

[att]
  [W_query]
    weight: 576 parameters
  [W_key]
    weight: 576 parameters
  [W_value]
    weight: 576 parameters


In [194]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    
    for batch, (X, y) in enumerate(dataloader):

        logits = model(X)
        loss = loss_fn(logits.flatten(0, 1), y.flatten())
       
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"Batch: {batch+1}, Loss: {loss:>7f}")

In [195]:
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    print("-----------------------------------")
    print("DONE!")

Epoch 1
Batch: 1, Loss: 5.374989
Batch: 2, Loss: 5.302770
Batch: 3, Loss: 4.787318
Batch: 4, Loss: 4.722095
-----------------------------------
DONE!
Epoch 2
Batch: 1, Loss: 4.975405
Batch: 2, Loss: 4.751341
Batch: 3, Loss: 4.474388
Batch: 4, Loss: 4.808431
-----------------------------------
DONE!
Epoch 3
Batch: 1, Loss: 4.534970
Batch: 2, Loss: 4.406578
Batch: 3, Loss: 4.550091
Batch: 4, Loss: 4.552303
-----------------------------------
DONE!
Epoch 4
Batch: 1, Loss: 4.393786
Batch: 2, Loss: 4.104616
Batch: 3, Loss: 4.526597
Batch: 4, Loss: 4.201035
-----------------------------------
DONE!
Epoch 5
Batch: 1, Loss: 4.469867
Batch: 2, Loss: 4.047516
Batch: 3, Loss: 4.006710
Batch: 4, Loss: 4.008577
-----------------------------------
DONE!
Epoch 6
Batch: 1, Loss: 3.728155
Batch: 2, Loss: 4.237046
Batch: 3, Loss: 3.896961
Batch: 4, Loss: 4.021441
-----------------------------------
DONE!
Epoch 7
Batch: 1, Loss: 3.857882
Batch: 2, Loss: 3.976342
Batch: 3, Loss: 3.716764
Batch: 4, Loss: 3

In [196]:
_=model.eval()

In [197]:
def generate(start_text):
    token_ids = text_to_token_ids(start_text, word_to_idx)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)


    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN:]

        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat([idx, idx_next], dim=1)

    idx = idx.view(-1).tolist()
    text = token_ids_to_text(idx, idx_to_word)
    return text    


In [198]:
text = generate("She composes")
print("Generated text: ", text)

Generated text:  She composes songs and curates art


In [199]:
text = generate("He reads")
print("Generated text: ", text)

Generated text:  He reads books and explores the


In [201]:
file_name = "parameters.bin"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, file_name))